# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Name: ", metadata.name)
print("Description: ", metadata.description)
print("Version: ", metadata.version)
print("Date Published: ", getattr(metadata, 'datePublished', ''))

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list all record sets, their IDs (`@id`), and included fields. 

According to the metadata, record sets can be accessed via their `@id`. Each record set contains fields, which also have their `@id`.

Below we enumerate all record sets and the fields within them.

In [ ]:
# List all record sets and their fields using their @id
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print("No record sets found in the Croissant metadata.\nTrying `dataset.record_sets` attribute (mlcroissant >=1.0.0).\n")
    record_sets = getattr(dataset, 'record_sets', [])

record_set_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    fields = rs.get('field', [])
    print("Fields:")
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        if isinstance(field, dict):
            print(f"  - {field.get('@id', 'Unknown')} (name: {field.get('name', 'N/A')})")
        else:
            # field might be an @id string reference
            print(f"  - {field}")
    print('-'*40)
if not record_set_ids:
    print("No record sets found. Please check the schema or library version.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract each available record set into a pandas DataFrame. You may inspect the column list for further analysis.

In [ ]:
# If the record_set_ids is empty, manually specify the record set IDs based on prior metadata inspection.
if not record_set_ids:
    # Here, provide a fallback if the record_set field is empty (reference schema to fill these in if needed)
    record_set_ids = [
        'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034' # Hypothetical value, replace with the correct one after inspection
    ]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}:\n", df.columns.tolist())
    display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

Replace `<numeric_field_id>` and `<group_field_id>` with actual field `@id` values obtained from the previous overview. If record set content is unknown, adapt as needed.

In [ ]:
# ---
# Example using the first record set loaded in the previous cell.
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Inspect columns for numeric candidates
    print("Available columns:", df.columns.tolist())
    # Suppose 'abundance' is a numeric field (replace with actual field @id from your overview)
    # If needed, print the head for exploration
    display(df.head())

    # Set these to valid column names (preferably field @id strings)
    numeric_field = None
    for c in df.columns:
        # Try to auto-detect a likely numeric field
        if 'abundance' in c.lower() or 'count' in c.lower() or 'mw' in c.lower():
            numeric_field = c
            break
    # Fallback: take the first column that is numeric
    if not numeric_field:
        for c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field = c
                break
    if not numeric_field:
        print("No numeric field found! Please choose a relevant field.")
    else:
        print(f"Using numeric field for EDA: {numeric_field}")
        # Convert to numeric (if not already)
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

        # Filter for values greater than threshold (e.g., 10)
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Group by a categorical field, e.g., 'description' or field with few unique values
        possible_group_fields = [c for c in df.columns if c != numeric_field and df[c].nunique() < max(10, len(df)//10)]
        group_field = possible_group_fields[0] if possible_group_fields else None

        if group_field:
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
else:
    print("No dataframes were created - check earlier steps.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we show an example histogram and scatter plot if numeric and group fields are available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    df = dataframes[record_set_id]
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded a proteomics dataset using its Croissant metadata with `mlcroissant`.
- Explored available record sets and fields using `@id` references, ensuring reproducibility.
- Loaded data into pandas, filtered and normalized a selected numeric field, and visualized field distributions with histograms and boxplots.

Next steps:
- Investigate additional record sets if present.
- Perform deeper analysis on modified proteins, peptide counts, or normalized abundances as appropriate.

Refer to the Croissant schema or documentation for more details regarding each record set and field `@id`.